# Activity 3: Isolation Forest and ML Based Detection

**Week 6 Day 2 · Machine Learning for Data Cleaning**

## Where we left off

Activity 2 ended with a working solution and an uncomfortable caveat.

You detected all five labelled taxi events, but only after **you** decided that the right reference population was "the same weekday at the same time of day", and only after you built that reference by hand with a `groupby`. One column, one seasonal pattern, one correct answer that you happened to guess.

Real tables are not like that. A claims table has dozens of columns, and the anomalies live in combinations nobody thought to check in advance.

This activity is about the family of methods that **learn the structure themselves**. The headline algorithm is Isolation Forest, which is the most widely deployed general purpose anomaly detector in production data systems today.

You are also going to find out where it fails, which matters more than knowing where it succeeds.

## Learning objectives

By the end of this activity you will be able to:

1. Explain how Isolation Forest detects anomalies, and why it inverts the usual approach.
2. Use `contamination` correctly, and explain why it is a business input rather than a tuned hyperparameter.
3. Work with anomaly **scores** rather than only binary labels.
4. Demonstrate that Isolation Forest and Mahalanobis distance detect **different kinds** of anomaly, and neither dominates.
5. Show that feature engineering changes results far more than hyperparameter tuning does.
6. Describe where Isolation Forest is and is not available across BigQuery, Snowflake, Spark, and Python.
7. Repair detected outliers by treating them as missing data, and judge when doing that is legitimate.

---
## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option("display.max_rows", 12)

This activity adds `scikit-learn` (`IsolationForest`) and `pyod` to the `matplotlib`, `scipy`, `pandas`, and `numpy` you already used in Activities 1 and 2. All of these are declared in the repo-root `pyproject.toml`, so `uv sync` from the repository root installs them. Run that once if you have not since they were added.

The next cell builds `DATA_DIR` the same way as the earlier activities: by walking up from wherever the notebook is running until it finds `pyproject.toml`, the marker for the repository root. That means the path resolves correctly whether you run this notebook from the course folder or from your own copy under `student-work/week6/day2/`, without you hardcoding it.

In [ ]:
from pathlib import Path


def find_repo_root(start=None):
    """Walk upward until we find the repo root (the folder holding pyproject.toml)."""
    start = start or Path.cwd()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repo root from " + str(start))


DATA_DIR = find_repo_root() / "Week 6" / "Labs" / "Day 2" / "data"

---
# Part 1: The idea behind Isolation Forest

Most detection methods build a model of what normal looks like, then measure how far each point sits from it. Mahalanobis distance does this. So does the Z-score.

Isolation Forest inverts the question. Instead of asking how far a point is from normal, it asks:

> **How hard is this point to separate from everything else?**

The procedure is almost absurdly simple. Pick a random column. Pick a random split value within its range. Divide the data. Repeat until every point sits alone in its own region.

Points in dense regions need many splits to isolate, because they are surrounded by neighbours. Points sitting alone in sparse regions get cut off almost immediately.

**The anomaly score is the average number of splits needed to isolate a point.** Fewer splits means more anomalous.

Build the intuition on a one dimensional example.

In [ ]:
rng = np.random.default_rng(42)
normal_points = rng.normal(loc=50, scale=5, size=20)
demo = np.append(normal_points, 200.0)

print("Normal points range from", round(demo[:-1].min(), 1), "to", round(demo[:-1].max(), 1))
print("Plus one obvious anomaly at", demo[-1])

In [ ]:
def splits_to_isolate(values, target_index, rng, max_splits=50):
    """Randomly split the range until the target point is alone. Return the split count."""
    active = np.arange(len(values))
    for depth in range(1, max_splits + 1):
        subset = values[active]
        low, high = subset.min(), subset.max()
        if low == high:
            return depth
        cut = rng.uniform(low, high)
        if values[target_index] < cut:
            active = active[values[active] < cut]
        else:
            active = active[values[active] >= cut]
        if len(active) == 1:
            return depth
    return max_splits

Run that many times for the anomaly and for a typical point, then compare the averages.

In [ ]:
rng = np.random.default_rng(0)
anomaly_index = len(demo) - 1
typical_index = 10

anomaly_depths = [splits_to_isolate(demo, anomaly_index, rng) for _ in range(300)]
typical_depths = [splits_to_isolate(demo, typical_index, rng) for _ in range(300)]

print(f"Average splits to isolate the anomaly : {np.mean(anomaly_depths):.2f}")
print(f"Average splits to isolate a normal pt : {np.mean(typical_depths):.2f}")

The anomaly is isolated in roughly one or two splits. A typical point takes several times as many.

That gap is the entire algorithm. Isolation Forest builds many such random trees and averages the isolation depth across all of them.

Three consequences fall directly out of this design, and they explain why it is so widely used:

- **No distance calculations.** It never computes how far apart any two points are, so it stays fast in high dimensions where distance based methods degrade.
- **No distributional assumptions.** It never assumes normality, unlike the Z-score.
- **Sub-linear cost.** Each tree is built from a small subsample, typically 256 points, so training time barely grows with dataset size.

---
# Part 2: Isolation Forest in scikit-learn

Return to the height and weight data from Activity 2.

In [ ]:
people = pd.read_csv(DATA_DIR / "weight-height.csv")
features = people[["Height", "Weight"]].to_numpy()
print(people.shape)

In [ ]:
from sklearn.ensemble import IsolationForest

In [ ]:
detector = IsolationForest(
    n_estimators=200,
    contamination=0.001,
    random_state=42,
)
detector.fit(features)

## The `contamination` parameter, and the mistake everyone makes

`contamination` is the expected proportion of anomalies. It does not change how the forest scores points at all. It only sets **where the cutoff is drawn** on those scores.

That has an important implication. The model will always flag approximately that fraction of your data, whether or not anything is actually wrong.

In [ ]:
for rate in [0.001, 0.005, 0.01, 0.05]:
    flagged = (IsolationForest(contamination=rate, random_state=42).fit_predict(features) == -1).sum()
    print(f"contamination = {rate:<6} flags {flagged:>4} rows ({flagged / len(people):.2%})")

Set it to 0.05 and you get 500 anomalies. Set it to 0.001 and you get 10. The data never changed.

**`contamination` is not a hyperparameter you tune for accuracy.** There is no validation score to optimise, because in unsupervised detection you have no labels to score against. It is a statement about how many records you are willing to investigate.

The correct way to choose it is a capacity conversation. If one analyst can review 50 flagged records a day, then on a 100,000 row daily feed your contamination is 0.0005. Work backwards from the review budget, not forwards from a default.

### The predict convention

One detail that causes real bugs: scikit-learn returns `-1` for anomalies and `1` for normal points. This is inverted from the usual convention where 1 means "positive detection".

In [ ]:
predictions = detector.predict(features)
print("Unique values returned by predict():", np.unique(predictions))

people["is_anomaly"] = predictions == -1
print("Flagged:", people["is_anomaly"].sum())

## Scores are more useful than labels

The binary label throws away information. `score_samples` gives the underlying continuous score, where **lower means more anomalous**.

In [ ]:
people["anomaly_score"] = detector.score_samples(features)
people["anomaly_score"].describe().round(4)

In production, prefer the score. It lets you rank records so reviewers see the worst cases first, adjust the threshold later without retraining, and monitor for drift by watching the score distribution over time.

A pipeline that stores only `True` or `False` has discarded the information needed to do any of that.

In [ ]:
worst = people.nsmallest(10, "anomaly_score")
worst[["Gender", "Height", "Weight", "anomaly_score"]].round(2)

Look at what it found: the tallest and heaviest men, and the shortest and lightest women. The extreme corners of the data cloud.

Hold onto that observation. It is about to matter.

---
# Part 3: Isolation Forest versus Mahalanobis, an honest comparison

In Activity 2, Mahalanobis distance found 4 anomalies that no univariate method could see: people whose height and weight were individually normal but jointly implausible, such as 5 feet 3 inches at 185 pounds.

Isolation Forest is the more modern, more popular, more general method. Check whether it found them too.

In [ ]:
centre = features.mean(axis=0)
inv_covariance = np.linalg.inv(np.cov(features.T))
deviations = features - centre
mahalanobis = np.sqrt(np.einsum("ij,jk,ik->i", deviations, inv_covariance, deviations))
maha_flags = mahalanobis > np.sqrt(stats.chi2.ppf(0.999, df=2))

print("Mahalanobis flagged:", maha_flags.sum())

In [ ]:
iforest_flags = people["is_anomaly"].to_numpy()

print("Isolation Forest flagged :", iforest_flags.sum())
print("Mahalanobis flagged      :", maha_flags.sum())
print("Flagged by BOTH          :", (iforest_flags & maha_flags).sum())

**They agree on nothing.**

Two respected multivariate anomaly detectors, run on the same 10,000 rows, produced completely disjoint answers. This is not a bug, and it is the most useful thing in this notebook.

See it directly.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6.5))
ax.scatter(people["Height"], people["Weight"], s=6, alpha=0.18, color="lightsteelblue", label="Normal")
ax.scatter(people.loc[iforest_flags, "Height"], people.loc[iforest_flags, "Weight"],
           s=70, color="tab:green", marker="s", label="Isolation Forest", zorder=3)
ax.scatter(people.loc[maha_flags, "Height"], people.loc[maha_flags, "Weight"],
           s=140, facecolors="none", edgecolors="tab:red", linewidths=2,
           label="Mahalanobis", zorder=4)
ax.set_xlabel("Height (inches)")
ax.set_ylabel("Weight (pounds)")
ax.set_title("Two detectors, two completely different definitions of anomalous")
ax.legend()
plt.tight_layout()
plt.show()

The chart explains everything.

**Isolation Forest (green) found the far corners.** Points at the extreme ends of the cloud, easy to isolate because nothing surrounds them. They sit *on* the correlation line, just a long way out. These are perfectly ordinary very tall people and very short people.

**Mahalanobis (red) found points off the diagonal.** They sit comfortably inside the overall range, in a reasonably populated area, but they violate the height-to-weight relationship the population follows.

The reason comes down to mechanics. Isolation Forest splits **one column at a time**, along axis parallel cuts. It measures sparsity in the raw feature space. A point can be well inside both the height range and the weight range, so no single axis parallel cut isolates it quickly, even when the pairing is impossible. Mahalanobis explicitly uses the covariance matrix, so violating the correlation is exactly what it measures.

### Which one is right?

Neither. They answer different questions, and the question you want depends on the failure you are trying to catch.

| You want to catch | Use | Because |
| :--- | :--- | :--- |
| Extreme values, sensor spikes, impossible magnitudes | Isolation Forest | Finds sparse regions, scales to many columns |
| Broken relationships between fields, transposed or mis-mapped columns | Mahalanobis | Explicitly models covariance |
| Both | Both, and take the union | They genuinely do not overlap |

The practical takeaway for a data engineer: **do not assume the newer or more popular algorithm supersedes the older one.** Run more than one detector, compare, and treat disagreement as information about your data rather than a problem to resolve.

If this had been a claims pipeline and you had deployed only Isolation Forest, the four records with impossible field combinations would have loaded silently into your warehouse.

---
# Part 4: The thing that actually determines your results

Take Isolation Forest to the taxi data, where you have labelled events to score against.

Start with the obvious features: the value, and the time information.

In [ ]:
taxi = pd.read_csv(DATA_DIR / "nyc_taxi.csv", parse_dates=["timestamp"]).set_index("timestamp")

KNOWN_EVENTS = {
    "2014-11-02": "NYC Marathon",
    "2014-11-27": "Thanksgiving",
    "2014-12-25": "Christmas",
    "2015-01-01": "New Year's Day",
    "2015-01-27": "North American Blizzard",
}

In [ ]:
frame = taxi.copy()
frame["weekday"] = frame.index.dayofweek
frame["time_of_day"] = frame.index.hour * 60 + frame.index.minute

In [ ]:
def detect_and_report(data, feature_names, label, contamination=0.02):
    """Fit Isolation Forest on the given features and report detections per known event."""
    model = IsolationForest(
        n_estimators=200, contamination=contamination, random_state=42
    ).fit(data[feature_names])

    flagged = model.predict(data[feature_names]) == -1
    per_day = pd.Series(flagged, index=data.index).groupby(data.index.date).sum()

    print(f"--- {label}")
    print(f"    features: {feature_names}")
    print(f"    flagged slots on a typical day (median): {int(per_day.median())}")
    for date, name in KNOWN_EVENTS.items():
        count = int(per_day[pd.Timestamp(date).date()])
        marker = "detected" if count > 0 else "MISSED"
        print(f"      {name:26s} {count:>3} slots   {marker}")
    return per_day

In [ ]:
naive = detect_and_report(frame, ["value", "time_of_day", "weekday"], "Attempt 1: raw value and time")

Thanksgiving missed completely. Christmas barely registered.

The instinct at this point is to tune. Raise the contamination, add more trees, change the seed. Resist it, and think about what the model is being shown instead.

The features are `value`, `time_of_day`, and `weekday`. The model has to work out, entirely on its own, that 400,000 riders is normal at 3am but alarming at 6pm, and that the rules differ again on Sundays. It has to learn the entire seasonal structure from random axis parallel splits.

That is a lot to ask. So give it the comparison directly, exactly as you did in Activity 2.

In [ ]:
frame["expected"] = frame.groupby(["weekday", "time_of_day"])["value"].transform("median")
frame["residual"] = frame["value"] - frame["expected"]

In [ ]:
informed = detect_and_report(frame, ["residual"], "Attempt 2: the residual")

**All five events detected**, and Thanksgiving went from 0 flagged slots to 15.

Compare what changed between the two runs:

- Same algorithm.
- Same `contamination`.
- Same `n_estimators`.
- Same `random_state`.
- **Fewer features than before**, not more.

The only difference is that the second version was given a feature encoding the right comparison. Check that the flagged days are real events rather than noise.

In [ ]:
informed.sort_values(ascending=False).head(10)

The blizzard, New Year's Day, Christmas, Boxing Day, Thanksgiving. Nearly every top ranked day is a genuine event.

### The lesson

> **Feature engineering moved this result far more than any hyperparameter could.**

This is why data quality ML belongs to data engineers rather than being handed off. Choosing the right comparison, computing the residual, and getting the granularity right are all pipeline work. The `IsolationForest` call is three lines and barely matters by comparison.

A team that spends a week tuning the detector and no time on the features will lose to a team that does the reverse, every time.

---
# Part 5: The same algorithm in different places

You will not always be working in scikit-learn. It is worth knowing what is actually available where your data lives, because the answer is often "not this".

## PyOD

PyOD is a library offering more than 40 detectors behind one consistent interface. Its `IForest` wraps the scikit-learn implementation, so results match, but the API conventions differ in ways that cause bugs.

In [ ]:
from pyod.models.iforest import IForest

pyod_model = IForest(n_estimators=200, contamination=0.001, random_state=42)
pyod_model.fit(features)

In [ ]:
print("PyOD labels_ values     :", np.unique(pyod_model.labels_), "  (1 = anomaly, 0 = normal)")
print("sklearn predict values  :", np.unique(detector.predict(features)), " (-1 = anomaly, 1 = normal)")
print()
agreement = ((pyod_model.labels_ == 1) == (detector.predict(features) == -1)).mean()
print(f"Agreement between the two: {agreement:.1%}")

Identical detections, **opposite label conventions**. PyOD uses 1 for anomaly, scikit-learn uses -1. Mixing the two up inverts your entire result set while the code runs without error.

The score conventions are also flipped: PyOD's `decision_scores_` are higher for more anomalous points, while scikit-learn's `score_samples` are lower. Check the direction whenever you swap libraries.

PyOD's real value is being able to try many algorithms with no code changes.

In [ ]:
from pyod.models.ecod import ECOD

ecod = ECOD(contamination=0.001)
ecod.fit(features)
print("ECOD flagged:", int(ecod.labels_.sum()), "rows, with no parameters to tune")

ECOD is worth knowing as a baseline. It is deterministic, has essentially no parameters, and is often competitive with far more complex methods.

## What the cloud platforms actually offer

This is where expectations tend to break. Isolation Forest is the best known anomaly detection algorithm, so people assume it is available everywhere. It is not.

| Platform | Isolation Forest available? | What you use instead |
| :--- | :--- | :--- |
| scikit-learn | Yes, native `IsolationForest` | Not applicable |
| PyOD | Yes, `IForest` wrapping scikit-learn | 40+ other detectors behind one API |
| **BigQuery ML** | **No** | `ML.DETECT_ANOMALIES` over an autoencoder, k-means, PCA, or `ARIMA_PLUS` model |
| **Snowflake** | **No** | `SNOWFLAKE.ML.ANOMALY_DETECTION`, a gradient boosting forecaster that compares actuals against predictions |
| **Spark MLlib** | **No native implementation** | scikit-learn inside a pandas UDF, or a third party package such as LinkedIn's `isolation-forest` |

Read that table carefully, because it changes how you design a pipeline.

In BigQuery, anomaly detection is framed around models you already have. Train k-means and anomalies are points far from every centroid. Train `ARIMA_PLUS` and anomalies are points outside the forecast interval. Different mechanism, same job, and it runs in SQL next to your data.

Snowflake's approach is explicitly forecast based. It predicts what the value should have been and flags large deviations, which suits time series and does not directly apply to unordered tabular rows.

Spark has no built in option at all, which surprises people building large scale pipelines.

### The decision this forces

You have to choose between two designs:

1. **Move the compute to the data.** Use the platform's native method, accept a different algorithm, keep everything in SQL, and avoid exporting rows.
2. **Move the data to the compute.** Pull it into Python, run exactly the algorithm you want, and pay for the export, the orchestration, and a second system to operate.

For a 50 million row table in BigQuery, exporting to a Python process to run Isolation Forest is usually the wrong call. `ML.DETECT_ANOMALIES` over a k-means model is not the algorithm you originally wanted, but it runs where the data already is.

"Which algorithm is best" is the wrong opening question. **"Which algorithm is best among those available where my data lives"** is the engineering question, and it frequently has a different answer.

---
# Part 6: Closing the loop, outliers as missing data

Now connect this back to Activity 1.

Once a value is flagged as an outlier, you have three options:

1. **Keep it and flag it.** Add a boolean column and let downstream consumers decide.
2. **Drop the row.** Simple, and it silently biases every aggregate computed afterwards.
3. **Treat the value as missing and impute it.** Erase the suspect number, then use the imputation machinery from Activity 1 to reconstruct a plausible one.

Option 3 is the one people overlook, and it is often the best choice. It preserves the row, so counts and joins stay intact, while removing a value you do not trust.

The mechanics are simple.

In [ ]:
model = IsolationForest(n_estimators=200, contamination=0.02, random_state=42)
frame["is_outlier"] = model.fit_predict(frame[["residual"]]) == -1

print("Flagged slots:", int(frame["is_outlier"].sum()), "out of", len(frame))

In [ ]:
frame["value_masked"] = frame["value"].mask(frame["is_outlier"])
print("Values now missing:", int(frame["value_masked"].isna().sum()))

In [ ]:
frame["value_repaired"] = frame["value_masked"].interpolate(method="time")
print("Values still missing after interpolation:", int(frame["value_repaired"].isna().sum()))

Three lines: detect, mask, impute. The outlier handling problem became the missing data problem you already know how to solve, and how to score.

Look at what it did to the blizzard.

In [ ]:
window = frame.loc["2015-01-26":"2015-01-29"]

fig, ax = plt.subplots(figsize=(13, 4.5))
ax.plot(window.index, window["value"], color="lightgray", lw=3, label="Original")
ax.plot(window.index, window["value_repaired"], color="tab:blue", lw=1.4, label="Repaired")
ax.scatter(window.index[window["is_outlier"]], window.loc[window["is_outlier"], "value"],
           color="tab:red", s=18, zorder=3, label="Flagged as outlier")
ax.set_ylabel("Passengers per 30 minutes")
ax.set_title("The blizzard, before and after automated repair")
ax.legend()
plt.tight_layout()
plt.show()

## Now stop and look at what you just did

The pipeline worked exactly as designed. It found the anomalous readings and replaced them with smooth, plausible, entirely fictional values.

**It erased the blizzard.**

That storm shut New York City down. It is the single most interesting event in this dataset, it is real, and the numbers were completely correct. An automated cleaning step just replaced them with an estimate of what an ordinary Tuesday would have looked like.

This is the judgment that separates a data engineer from someone running library calls.

| Situation | Repair? | Why |
| :--- | :--- | :--- |
| Sensor reported -999 | Yes | Not a measurement, it is a failure code |
| Meter reported 10x its physical maximum | Yes | Physically impossible, so it is corrupt |
| Feed went flat for six hours | Yes | Collection broke, the values are not real |
| Ridership collapsed during a blizzard | **No** | Real event, correct data, and erasing it destroys history |
| Claim amount 50x the usual | **No** | Possibly fraud, possibly a catastrophe claim, and either way somebody needs to see it |

The pattern: **repair values that are wrong, preserve values that are merely surprising.**

An outlier detector cannot tell you which case you are in. It measures statistical unusualness, and nothing else. A broken sensor and a genuine emergency look identical to it, and both are unusual by construction.

So the rule is: automate detection, keep a human in the loop on repair, and never destroy the original.

In [ ]:
audit_ready = frame[["value", "value_repaired", "is_outlier"]].copy()
audit_ready = audit_ready.rename(columns={"value": "value_original"})
audit_ready["was_repaired"] = audit_ready["is_outlier"] & audit_ready["value_original"].notna()

audit_ready.loc[audit_ready["was_repaired"]].head()

Keep the original column, keep the repaired column, and keep the flag. Storage is cheap. Being unable to answer "what did the raw feed actually say" during an incident is not.

---
# Your Turn

Work in your own copy under `student-work/week6/day2/`.

## Challenge 1: Set contamination from a budget

Your team can review 25 flagged records per day. The claims feed delivers 40,000 records daily.

1. Calculate the contamination rate that produces roughly that review volume.
2. Fit Isolation Forest on the height and weight data using it.
3. Rank the flagged records by `score_samples` and show the 10 that a reviewer would see first.
4. In a markdown cell, explain why ranking matters even when the flag count is already within budget.

## Challenge 2: Reproduce the disagreement

Part 3 showed Isolation Forest and Mahalanobis flagging entirely different rows.

1. Re-run both at three contamination levels: 0.001, 0.005, and 0.01.
2. For each level, report how many rows both methods flagged.
3. Determine whether the overlap ever becomes substantial.
4. Build a combined detector that flags the union of the two, and describe one pipeline where you would want that and one where you would not.

## Challenge 3: Prove the features matter more

Part 4 claimed feature engineering beats hyperparameter tuning. Test it properly.

1. Using only `["value", "time_of_day", "weekday"]`, try at least six combinations of `n_estimators` and `contamination`.
2. Record how many of the five known events each configuration detects.
3. Report the best result any tuning achieved.
4. Compare against the single feature `["residual"]` run at default settings.
5. State in a markdown cell whether tuning ever closed the gap.

## Challenge 4: Decide what to repair

Using the flagged slots from Part 6:

1. Group the flagged 30 minute slots by date.
2. Separate days with many flagged slots from days with only one or two isolated flags.
3. Argue which group is more likely to be genuine events and which is more likely to be collection faults.
4. Write a repair rule that acts on one group and leaves the other alone, and implement it.

State your reasoning explicitly. The rule matters less than being able to defend it.

## Challenge 5: Choose a platform, and justify it

You are designing anomaly detection for a claims table of 80 million rows already in BigQuery. Results must refresh nightly and feed a Looker dashboard.

Using the platform table in Part 5, write a recommendation of no more than 250 words covering:

- whether you would run detection in BigQuery or export to Python
- which algorithm that choice commits you to
- what you give up by not using Isolation Forest
- what would have to change for you to switch approaches

Write it for a technical lead who has to approve the design.

---
## What you did

- Built the intuition for Isolation Forest by measuring isolation depth directly.
- Established that `contamination` sets a cutoff, not a detection quality, and should be driven by review capacity.
- Used continuous scores instead of binary labels, and explained why production pipelines need them.
- Found that Isolation Forest and Mahalanobis flagged **completely disjoint** sets of rows, and explained the mechanism.
- Demonstrated that swapping in one engineered feature outperformed every hyperparameter change.
- Mapped where Isolation Forest is actually available across BigQuery, Snowflake, Spark, and Python.
- Repaired outliers using imputation, then recognised that the pipeline had erased a real blizzard.

## The theme of the day so far

Across all three activities the same discipline keeps paying off:

1. Look at the data before choosing a method.
2. Compute the trivial baseline first.
3. Measure against something real rather than trusting reputation.
4. When methods disagree, investigate rather than pick.
5. Know what the pipeline threw away before blaming the algorithm.

Next: **Activity 4**, where AutoML runs dozens of models for you, and where you decide what is worth keeping when the tool does the work.